# 💼 Korean Corporate Credit Survival Analysis

> 한국 기업의 부도위험을 Kaplan-Meier → Cox PH → Random Survival Forest로 다층 분석.

**Research Question**: 재무비율(ROA, 부채비율, 유동비율)이 5년 부도확률에 어떻게 영향을 미치는가?

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from credit_surv.models import KaplanMeier, CoxModel, RSFModel, compare_survival_models
from credit_surv.data_loader import load_synthetic_credit_panel

## 1. 데이터 로드

In [ ]:
df = load_synthetic_credit_panel(n_firms=2000, censoring_rate=0.6, seed=42)
print(f'Shape: {df.shape}')
print(f'Event rate: {df.event.mean():.2%}')
df.head()

## 2. Kaplan-Meier — 비모수 생존곡선

In [ ]:
km = KaplanMeier()
km.fit(df['duration'], df['event'], label='전체')
print(f'중위 생존시간: {km.median_survival_time():.0f}일')

# 산업별 비교
fig, ax = plt.subplots(figsize=(10, 6))
for ind in df['industry'].unique():
    sub = df[df.industry == ind]
    KaplanMeier().fit(sub.duration, sub.event, label=ind).survival_function().plot(ax=ax)

## 3. Cox PH — 위험비(Hazard Ratio) 추정

In [ ]:
cox = CoxModel(duration_col='duration', event_col='event')
cox_result = cox.fit(df, covariates=['roa', 'debt_ratio'])
print(cox_result.summary())
# 예: HR(debt_ratio) = 1.65 → 부채비율 1단위 ↑ = 부도 위험 65% ↑

## 4. 비례위험 가정 검정

In [ ]:
ph_check = cox.check_proportional_hazards()
ph_check

## 5. Random Survival Forest — 비선형 + 상호작용

In [ ]:
rsf = RSFModel(n_estimators=100, max_depth=5)
rsf_result = rsf.fit(df[['roa', 'debt_ratio']], df[['duration', 'event']])
print(f'C-index: {rsf_result.concordance_index:.3f}')
rsf.feature_importance().plot.barh()

## 6. 모형 비교

In [ ]:
comparison = compare_survival_models([cox_result, rsf_result])
comparison

## 7. 결론

- Cox PH: 해석력 ↑, 비례위험 가정 검증 필요
- RSF: 예측력 ↑, SHAP으로 비선형 효과 시각화
- **실무 권장**: Cox PH로 인사이트 + RSF로 스코어링 시스템 구축